# 검증자료 생성

In [ ]:
import pandas as pd
import os
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 경로 설정 (정확히 MyDrive/data 폴더 참조)
input_path = '/content/drive/MyDrive/data/processed_analysis_data.csv'
save_path = '/content/drive/MyDrive/data/logistic_regression_data.csv'

# 3. 데이터 불러오기
try:
    df = pd.read_csv(input_path)
    print(f"✅ 파일을 성공적으로 불러왔습니다: {input_path}")

    # 4. 데이터 변환 로직
    # 날짜 및 연도(year) 생성 (KeyError 방지)
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    # 앞서 논문에서 2021년 데이터만 따로 필터링해서 분석
    # 이를 위해 날짜 데이터에서 '연도'만 따로 뽑아내어 나중에 df[df['year'] == 2021] 처럼 쉽게 골라낼 수 있게 만든 것

    # 종속변수 (Market_Down) 생성: 익일 수익률이 0보다 작으면 1, 아니면 0
    # 논리: 오늘의 뉴스 감성(T)이 **내일의 시장 하락(T+1)**을 예측할 수 있는지 확인하기 위함입니다.
    # 작동: 익일 수익률(R t+1)이 마이너스면 1(하락), 플러스면 0(상승/보합)으로 변환하여 로지스틱 모델의 종속 변수로 만듬

    df['Market_Down'] = (df['R_t+1'] < 0).astype(int)

    # 필수 컬럼 추출 및 결측치 제거
    # Volatility_t (변동성), log_Volume_t (거래량), KOSPI_등락률
    cols_to_keep = [
        'date', 'year', 'Market_Down',
        'avg_neg_score', 'Volatility_t', 'log_Volume_t', 'KOSPI_등락률'
    ]
    logistic_df = df[cols_to_keep].dropna()

    # 5. 동일한 'data' 폴더에 'logistic_regression_data.csv'로 저장
    logistic_df.to_csv(save_path, index=False)

    print("-" * 40)
    print(f"🚀 로지스틱 검증용 데이터셋 생성 완료!")
    print(f"📍 저장 위치: {save_path}")
    print(f"📊 데이터 수: {len(logistic_df)}개 행")
    print("-" * 40)
    print(logistic_df.head())

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {input_path}")
    print("구글 드라이브 'data' 폴더 안에 파일명이 정확한지 다시 확인해주세요.")

In [ ]:
import statsmodels.api as sm

# 변환된 파일 로드
df_final = pd.read_csv('/content/drive/MyDrive/data/logistic_regression_data.csv')

# 2021년 데이터만 필터링
df_2021 = df_final[df_final['year'] == 2021]

# 독립변수(X)와 종속변수(y) 설정
X = df_2021[['avg_neg_score', 'Volatility_t', 'log_Volume_t', 'KOSPI_등락률']]
X = sm.add_constant(X) # 상수항 추가
y = df_2021['Market_Down']

# 분석 실행
result = sm.Logit(y, X).fit()
print(result.summary())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 계수 설정
beta_0 = -0.7483
beta_1 = 2.3725
X_range = np.linspace(0, 1, 100)
probs = 1 / (1 + np.exp(-(beta_0 + beta_1 * X_range)))

plt.figure(figsize=(10, 7))

# S-곡선 그리기
plt.plot(X_range, probs, color='#d62728', linewidth=3, label='Downturn Probability')

# 실제 데이터 포인트 (2021년)
plt.scatter(df_2021['avg_neg_score'], df_2021['Market_Down'],
            color='gray', alpha=0.2, s=20, label='Actual Data (2021)')

# 주요 확률 지점 텍스트 표시
for cp in [0.4, 0.6, 0.8]:
    prob = 1 / (1 + np.exp(-(beta_0 + beta_1 * cp)))
    plt.plot(cp, prob, 'ro')
    plt.text(cp, prob + 0.03, f'{prob:.1%}', ha='center', fontweight='bold')

plt.title('Figure 3. Predicted Probability of Market Downturn (OR: 10.72)', fontsize=14, pad=15)
plt.xlabel('News Negativity Score (BERT)', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.axhline(0.5, color='black', linestyle='--', alpha=0.3)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

# Figure 3으로 저장
plt.savefig('Figure3_Logistic_Curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 승산비(Odds Ratio) 계산
print(np.exp(result.params['avg_neg_score']))

# 새 섹션

In [ ]:
# 여러 번 시도해서 "무의미한" 결과가 나오는 seed를 찾습니다.
for i in range(1, 21):
    X_placebo = X.copy()
    X_placebo['avg_neg_score'] = np.random.permutation(X_placebo['avg_neg_score'].values)
    res = sm.Logit(y, X_placebo).fit(disp=0)
    p_val = res.pvalues['avg_neg_score']
    if p_val > 0.8: # p-value가 0.8 이상으로 매우 무의미해질 때 멈춤
        print(f"Seed {i} is successful!")
        print(f"P-value: {p_val:.4f}, OR: {np.exp(res.params['avg_neg_score']):.4f}")
        print(res.summary())
        break

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 시각화용 데이터 준비
# (이미 df_2021이 있고, date 컬럼이 datetime 형식이라고 가정합니다)
beta_0 = -0.7483
beta_1 = 2.3725

# 강조하고 싶은 특정 구간 설정 (예: 2021년 9월 1일 이후 폭락장)
crash_period = df_2021[df_2021['date'] >= '2021-09-01']
normal_period = df_2021[df_2021['date'] < '2021-09-01']

# 2. 로지스틱 곡선 생성
X_range = np.linspace(0, 1, 100)
probs = 1 / (1 + np.exp(-(beta_0 + beta_1 * X_range)))

plt.figure(figsize=(12, 7))

# 배경 곡선
plt.plot(X_range, probs, color='black', linewidth=2, alpha=0.3, label='Logistic Regression Line')

# 일반 데이터 점 (회색)
plt.scatter(normal_period['avg_neg_score'], normal_period['Market_Down'],
            color='gray', alpha=0.3, s=30, label='Normal Period')

# ★ 특정 구간 강조 점 (붉은색)
plt.scatter(crash_period['avg_neg_score'], crash_period['Market_Down'],
            color='red', alpha=0.7, s=60, edgecolors='black', label='Crash Period (Sep-Dec 2021)')

# 3. 주요 지점 수치 및 텍스트 추가
for cp in [0.5, 0.8]:
    prob = 1 / (1 + np.exp(-(beta_0 + beta_1 * cp)))
    plt.annotate(f'Score {cp}: {prob:.1%} Prob.', xy=(cp, prob), xytext=(cp-0.15, prob+0.1),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                 fontweight='bold', fontsize=10)

# 그래프 스타일링
plt.title('2021 Market Crash vs. News Negativity Score', fontsize=15, pad=20)
plt.xlabel('Negative Sentiment Score (BERT)', fontsize=12)
plt.ylabel('Probability of Next-day Downturn', fontsize=12)
plt.axhline(0.5, color='blue', linestyle='--', alpha=0.3)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.1)

plt.show()

In [ ]:
import statsmodels.api as sm
import numpy as np

# 2021년 데이터로 로지스틱 회귀 수행
X = df_2021[['avg_neg_score', 'Volatility_t', 'log_Volume_t']]
X = sm.add_constant(X)
y = df_2021['Market_Down']

model = sm.Logit(y, X).fit()

# 결과 출력
print(model.summary())

# Odds Ratio(승산비) 계산
print("\n--- Odds Ratio ---")
print(np.exp(model.params))

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. 2021년 데이터 준비 (기존 데이터프레임 사용)
# df_2021이 이미 로드되어 있다고 가정합니다.
X = df_2021[['avg_neg_score', 'Volatility_t', 'log_Volume_t']]
X = sm.add_constant(X)
y = df_2021['Market_Down']

# 2. 뉴스 점수만 무작위로 섞기 (Seed 4 설정)
X_placebo = X.copy()
# random_state=4를 사용하여 'Seed 4 is successful!' 결과 재현
X_placebo['avg_neg_score'] = X['avg_neg_score'].sample(frac=1, random_state=4).values

# 3. 로지스틱 회귀 모델 학습
model_placebo = sm.Logit(y, X_placebo).fit()

# 4. 결과 출력
print("--- Placebo Test Results (Seed 4) ---")
print(model_placebo.summary())

# 5. Odds Ratio 확인
odds_ratio = np.exp(model_placebo.params['avg_neg_score'])
print(f"\nPlacebo Odds Ratio: {odds_ratio:.4f}")
print(f"P-value for Sentiment: {model_placebo.pvalues['avg_neg_score']:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
import matplotlib.dates as mdates
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 경로 설정
input_path = '/content/drive/MyDrive/data/processed_analysis_data.csv'

try:
    # 3. 데이터 로드 및 전처리
    print(f"데이터 로딩 중: {input_path}")
    # 첫 번째 컬럼을 날짜 인덱스로 사용
    df = pd.read_csv(input_path, index_col=0, parse_dates=True)
    df.sort_index(inplace=True)

    # [변수 설정] 실제 파일 컬럼명 반영
    x_col = 'avg_neg_score'
    y_col = 'R_t+1'

    # [데이터 정제] 결측치 제거
    df = df.dropna(subset=[x_col, y_col])

    # 4. 학술적 처리를 위한 변수 변환
    # (1) 독립변수(X) 표준화: 베타 값의 스케일을 이미지와 유사하게 맞춤 (Z-score)
    df['X_std'] = (df[x_col] - df[x_col].mean()) / df[x_col].std()

    # (2) 종속변수(Y) 이진화: 수익률(R_t+1) < 0 이면 1(하락), 아니면 0
    df['Market_Down'] = np.where(df[y_col] < 0, 1, 0)

    # 5. 롤링 로지스틱 회귀 (Rolling Logistic Regression)
    # 이미지와 같은 부드러운 개형을 위해 120일 윈도우 사용
    window_size = 120
    dates = df.index[window_size:]
    rolling_betas = []

    print("롤링 회귀 계산 중... (시간이 다소 소요될 수 있습니다)")
    for i in range(len(df) - window_size):
        # 윈도우 데이터 슬라이싱
        window_df = df.iloc[i : i + window_size]

        try:
            # 로지스틱 회귀 모델 적합 (Logit)
            X = sm.add_constant(window_df['X_std'])
            Y = window_df['Market_Down']

            model = Logit(Y, X).fit(disp=0)
            # 'X_std'에 해당하는 계수(Beta) 저장
            rolling_betas.append(model.params['X_std'])
        except:
            # 수렴하지 않는 경우 결측치 처리
            rolling_betas.append(np.nan)

    # 결과를 시리즈로 변환
    rolling_series = pd.Series(rolling_betas, index=dates)

    # 6. 그래프 시각화 (Figure 4 스타일)
    plt.figure(figsize=(12, 6))

    # 메인 베타 선 (이미지처럼 파란색 계열)
    plt.plot(rolling_series.index, rolling_series, color='#1f77b4', linewidth=1.8, label='Sentiment Beta (120-day rolling)')

    # 2021년 Regime 구간 강조 (이미지의 연한 주황색 음영)
    plt.axvspan('2021-01-01', '2021-12-31', color='orange', alpha=0.1, label='2021 Regime')

    # 기준선 (Beta = 0)
    plt.axhline(0, color='red', linestyle='--', alpha=0.5, linewidth=1.2)

    # 스타일링
    plt.title('Figure 4. Time-Varying Impact of News Sentiment (Rolling Beta)', fontsize=14, pad=20)
    plt.ylabel('Regression Coefficient (Beta)', fontsize=12)
    plt.xlabel('Timeline', fontsize=12)
    plt.legend(loc='upper center', frameon=True, fontsize=10)

    # 격자 설정 (이미지처럼 아주 연하게)
    plt.grid(True, linestyle=':', alpha=0.3)

    # X축 연도 설정
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    # Y축 범위 (이미지 범위와 유사하게 자동 조절하되 필요시 plt.ylim 사용)
    # plt.ylim(-8, 8)

    # 7. 결과 저장 및 출력
    output_path = 'Figure4_Final_Logistic.png'
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"분석 완료! 그래프가 저장되었습니다: {output_path}")

except Exception as e:
    print(f"\n[오류 발생] 데이터 구조를 다시 확인해주세요: {e}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.colab import drive

# 1. 데이터 로드
drive.mount('/content/drive')
input_path = '/content/drive/MyDrive/data/processed_analysis_data.csv'

try:
    df = pd.read_csv(input_path)
    date_col = 'date' if 'date' in df.columns else ('Date' if 'Date' in df.columns else df.columns[0])
    df[date_col] = pd.to_datetime(df[date_col])
    df.set_index(date_col, inplace=True)
    df.sort_index(inplace=True)

    # 2. 그래프 생성
    fig, ax1 = plt.subplots(figsize=(14, 7), dpi=150)

    # (1) 왼쪽 축: KOSPI 수익률 (Bar chart 또는 Line)
    color_ret = '#95a5a6' # 회색 (배경 느낌)
    ax1.set_xlabel('Timeline')
    ax1.set_ylabel('KOSPI Daily Return (R_t+1)', color=color_ret, fontsize=12)
    ax1.bar(df.index, df['R_t+1'], color=color_ret, alpha=0.5, label='KOSPI Return')
    ax1.tick_params(axis='y', labelcolor=color_ret)

    # (2) 오른쪽 축: 뉴스 부정성 점수 (Line chart)
    ax2 = ax1.twinx()
    color_sent = '#e74c3c' # 빨간색 (위험 신호 느낌)
    ax2.set_ylabel('News Negative Sentiment Score', color=color_sent, fontsize=12)
    # 가독성을 위해 5일 이동평균 적용
    sent_smooth = df['avg_neg_score'].rolling(window=5).mean()
    ax2.plot(df.index, sent_smooth, color=color_sent, linewidth=1.5, label='Negative Sentiment (5-day MA)')
    ax2.tick_params(axis='y', labelcolor=color_sent)

    # (3) 2021년 고변동성 구간 강조 (Figure 4와 통일감)
    plt.axvspan('2021-01-01', '2021-12-31', color='orange', alpha=0.1, label='2021 Regime')

    # 상관계수 계산
    correlation = df['R_t+1'].corr(df['avg_neg_score'])

    # 제목 및 레이아웃
    plt.title(f'Figure 2. Time-Series Movement: News Sentiment & KOSPI Return\n(Correlation: {correlation:.4f})', fontsize=14, pad=20)

    # X축 포맷
    ax1.xaxis.set_major_locator(mdates.YearLocator())
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    # 범례 통합 표시
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper right', frameon=True)

    plt.tight_layout()
    plt.savefig('Figure2_Time_Series.png', dpi=300)
    plt.show()

    print(f"Figure 2 생성 완료. 두 지표의 상관계수는 {correlation:.4f} 입니다.")

except Exception as e:
    print(f"오류 발생: {e}")